#КП2.3 Математика которая используется в статье
Броуновское движение, интеграл Ито, KL-дивергенция и все-все-все

### С математикой всё сложно: проблема в том что чтобы посчитать градиенты назад — нужно знать будущий шум. А будущий шум случайный и неизвестен.

Поэтому сначала авторы берут DLGM, устремиляют число слоёв к бесконечности, получают Neural SDE.\
Но чтобы сделать backpropagation идут обратно — дискретизуют Neural SDE с шагом Δt, получают обратно DLGM с N слоями. Только теперь все слои используют
одну и ту же нейросеть fθ вместо разных bi​.

Авторы так и пишут в Section 5.2:

> This approach approximates the continuous-time neural SDE by a discrete DLGM of the form

*5 Раздел статьи*

##5.2 Solve-then-differentiate: the Euler backprop из статьи

Itô SDE: определение и дискретизация

$$dX_t = f(X_t, t)\, dt + g(X_t, t)\, dW_t, \qquad X_0 = x_0$$

**Метод Эйлера** (используется в реализации):

$$X_{t_{k+1}} = X_{t_k} + f(X_{t_k}, t_k)\,\Delta t + g(X_{t_k}, t_k)\,\sqrt{\Delta t}\,\varepsilon_k, \qquad \varepsilon_k \sim \mathcal{N}(0, I)$$

##Класс, теперь когда мы умеем дискретизовать и дифференцировать SDE — возникает вопрос: **а что именно оптимизировать?**

Мы хотим чтобы модель хорошо объясняла данные $Y$, то есть максимизировать $\log p(Y)$. Но посчитать его напрямую нельзя — нужно проинтегрировать по всем траекториям. Поэтому используем нижнюю оценку — **ELBO**.

Вводим вариационный процесс $q_\theta$ (нейросеть как дрейф) и доказываем неравенство:

$$\log p(Y) \geq \underbrace{\mathbb{E}_{q_\theta}[\log p(Y|X_T)]}_{\text{reconstruction}} - \underbrace{\text{KL}(q_\theta \| p)}_{\text{регуляризация}}$$

Максимизируем правую часть — это и есть ELBO. Два слагаемых:

- **Reconstruction** — модель должна хорошо объяснять данные. В коде это moment matching — $\mathbb{E}[X_T]$ должно совпадать с $\mathbb{E}[Y]$
- **KL** — вариационный процесс не должен слишком сильно отклоняться от чистого броуновского движения. По теореме Гирсанова это $\frac{1}{2}\mathbb{E}\int_0^T \|f_\theta\|^2\,dt$ — штраф за большой дрейф. В коде это `kl_acc`

In [ ]:
# В коде:
def simulate_with_kl(net, n_samples):
    """
    Euler для вариационного процесса.
    Возвращает X_T и оценку KL(q_theta || Wiener).
    """
    X      = torch.zeros(n_samples, D, device=DEVICE)
    kl_acc = torch.zeros(1, device=DEVICE)

    for _ in range(N_STEPS):
        drift   = net(X)                                           # (n, D)
        kl_acc  = kl_acc + (drift ** 2).sum(dim=-1).mean() * DT * 0.5
        X       = X + drift * DT + SQRT_DT * torch.randn_like(X)

    return X, kl_acc

##**Итоговый теоритический смысл:**

Авторы показывают что предел существует и имеет красивую структуру. Neural SDE – это DLGM с бесконечным числом слоёв. Это даёт теоретическое обоснование почему такие модели мощные.

Переход к непрерывному времени даёт три вещи:

* Одна нейросеть вместо kk
k разных.В DLGM каждый слой bi​ свой. В Neural SDE одна fθ​ на всё — намного меньше параметров, можно брать сколько угодно шагов не увеличивая модель.

* Можно использовать умные SDE солверы которые сами выбирают размер шага — мелкий где нужна точность, крупный где можно сэкономить.
* Непрерывная формулировка даёт ELBO, теорему Гирсанова, KL в красивом виде — это проще анализировать и расширять.